# Notebook 07 — NLP: Sentiment Analysis & Topic Modelling

**Objective:** Analyse Kenya agricultural news for sentiment trends and topic clusters.

**Models:** VADER baseline → DistilBERT → BERTopic

**Note:** This is the stretch goal / X-factor module.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.utils import section

news = pd.read_csv("data/processed/news_clean.csv")
print("News shape:", news.shape)
print("Columns:", list(news.columns))
news.head(3)

## Step 1 — VADER Baseline Sentiment

In [ ]:
section("BASELINE — VADER Sentiment Analyser")
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

    analyser = SentimentIntensityAnalyzer()

    def get_sentiment(text):
        if not isinstance(text, str):
            return "Neutral"
        score = analyser.polarity_scores(text)["compound"]
        if score >= 0.05:
            return "Positive"
        elif score <= -0.05:
            return "Negative"
        return "Neutral"

    def get_compound_score(text):
        if not isinstance(text, str):
            return 0.0
        return analyser.polarity_scores(text)["compound"]

    news["vader_sentiment"] = news["title_clean"].apply(get_sentiment)
    news["vader_score"]     = news["title_clean"].apply(get_compound_score)

    print("Sentiment distribution:")
    print(news["vader_sentiment"].value_counts())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Distribution
    news["vader_sentiment"].value_counts().plot(
        kind="bar", ax=axes[0],
        color=["#66BB6A", "#BDBDBD", "#EF5350"],
        rot=0
    )
    axes[0].set_title("Sentiment Distribution (VADER)")
    axes[0].set_ylabel("Article Count")

    # Score distribution
    axes[1].hist(news["vader_score"], bins=20, color="#1565C0", alpha=0.7)
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].set_title("VADER Compound Score Distribution")
    axes[1].set_xlabel("Compound Score")

    plt.tight_layout()
    plt.savefig("reports/figures/11_vader_sentiment.png", dpi=150, bbox_inches="tight")
    plt.show()

except ImportError:
    print("Install: pip install vaderSentiment")

## Step 2 — BERTopic Topic Modelling

In [ ]:
section("TOPIC MODELLING — BERTopic")
try:
    from bertopic import BERTopic

    titles = news["title_clean"].dropna().tolist()
    print(f"Running BERTopic on {len(titles)} article titles...")

    topic_model = BERTopic(
        language="english",
        calculate_probabilities=True,
        verbose=False,
        nr_topics="auto",
        min_topic_size=5,
    )

    topics, probs = topic_model.fit_transform(titles)

    # Show topic info
    topic_info = topic_model.get_topic_info()
    print(f"\nFound {len(topic_info) - 1} topics:")
    print(topic_info[topic_info["Topic"] != -1].head(10).to_string(index=False))

    # Save topic model
    from src.models import save_model
    save_model(topic_model, "bertopic_news")

except ImportError:
    print("Install: pip install bertopic sentence-transformers")

## Step 3 — Sentiment Over Time

In [ ]:
section("SENTIMENT TREND — Monthly Average")

news["date_parsed"] = pd.to_datetime(news["date"], errors="coerce")
news["month_year"]  = news["date_parsed"].dt.to_period("M")

if "vader_score" in news.columns:
    monthly_sentiment = (
        news.groupby("month_year")["vader_score"]
        .agg(["mean", "count"])
        .reset_index()
    )
    monthly_sentiment["month_year"] = monthly_sentiment["month_year"].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    ax2 = ax1.twinx()

    ax1.bar(monthly_sentiment["month_year"], monthly_sentiment["count"],
            alpha=0.3, color="#1565C0", label="Article count")
    ax2.plot(monthly_sentiment["month_year"], monthly_sentiment["mean"],
             color="#E65100", linewidth=2, marker="o", label="Avg sentiment")
    ax2.axhline(0, color="grey", linestyle="--", alpha=0.5)

    ax1.set_xlabel("Month")
    ax1.set_ylabel("Article Count", color="#1565C0")
    ax2.set_ylabel("Average VADER Score", color="#E65100")
    plt.title("Monthly Agricultural News Sentiment (Kenya, 2025–2026)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("reports/figures/12_sentiment_over_time.png", dpi=150, bbox_inches="tight")
    plt.show()